In [1]:
import os
import gc
import copy
import torch
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
import time
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

import timm 
from torch import nn
from torch.optim import Adam
from torchvision import models
import torch.nn.functional as F
from torchvision.transforms import v2
from torchvision.datasets import ImageFolder
from torch.utils.data import Dataset, DataLoader, Subset, ConcatDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchinfo import summary
from typing import List, Tuple, Union
from PIL import Image

In [2]:
# Hyperparameters

BATCH_SIZE = 32
EPOCHS = 50
NUM_CLASSES = 3
DROPOUT_RATE = 0.3
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD = [0.229, 0.224, 0.225]

In [3]:
# Model name
MODEL1_NAME = 'DenseNet121'
MODEL2_NAME = 'MobileNetV2'
MODEL3_NAME = 'ResNet50'
MODEL4_NAME = 'VGG19'

In [4]:
# Enable cuDNN benchmark for optimal performance during inference profiling
torch.manual_seed(24520152)
torch.backends.cudnn.benchmark = True
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Inference profiling initialized on device: {DEVICE.upper()}")

Inference profiling initialized on device: CUDA


In [5]:
class DenseNet121(nn.Module):
    """
    DenseNet121-based model for image classification using Transfer Learning.
    
    Architecture:
    - Backbone: DenseNet121 (frozen, ImageNet weights)
    - Head: Global Average Pooling -> Dropout -> Linear (Dense)
    
    Attributes:
        num_classes (int): Number of target classes.
        dropout_rate (float): Dropout probability.
    """
    
    def __init__(self, num_classes: int = 3, dropout_rate: float = 0.3) -> None:
        super().__init__()

        # Load Pre-trained DenseNet121
        weights = models.DenseNet121_Weights.IMAGENET1K_V1
        backbone = models.densenet121(weights=weights)

        # DenseNet121 .features contains all Conv/Relu/MaxPool layers
        self.features = backbone.features

        # Unfreezing parameters for base evaluation. 
        for param in self.features.parameters():
            param.requires_grad = True  

        # Define Custom Classifier Head.
        self.in_features = 1024 
        self.global_avg_pool = nn.AdaptiveAvgPool2d(output_size=1) # Reduces (Batch, 1024, H, W) -> (Batch, 1024, 1, 1)
        self.flatten = nn.Flatten(start_dim=1) # Flatten (Batch, 1024, 1, 1) -> (Batch, 1024)
        self.dropout = nn.Dropout(p=dropout_rate)
        self.classifier = nn.Linear(in_features=self.in_features, out_features=num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass of the model.

        Args:
            x (torch.Tensor): Input image batch (B, C, H, W).
                              Expected standard ImageNet normalization.

        Returns:
            torch.Tensor: Raw logits (B, num_classes). 
                          NOTE: Does NOT apply Softmax.
        """
        
        # Feature extraction (Frozen)
        x = self.features(x)
        # The torchvision.models.densenet121 `.features` block ends with a 
        # BatchNorm layer (norm5), which outputs both negative and positive values.
        # We MUST apply ReLU here to zero out negative values (noise/background).
        x = F.relu(x, inplace=True)
        # Classification Head (Trainable)
        x = self.global_avg_pool(x)
        x = self.flatten(x)
        x = self.dropout(x)
        
        # Output Logits
        logits = self.classifier(x)
        
        return logits

In [6]:
def build_densenet121(num_classes: int = NUM_CLASSES, dropout_rate: float = DROPOUT_RATE) -> DenseNet121:
    """
    Factory function to instantiate the customized DenseNet121 model for Transfer Learning.

    This function initializes a `DenseNet121` which includes:
    1. A frozen DenseNet121 backbone (pre-trained on ImageNet).
    2. A custom trainable classification head (ReLU -> Pooling -> Dropout -> Linear).

    Args:
        num_classes (int, optional): The number of target classes for the classification task. 
            Determines the output size of the final Linear layer. Defaults to 3.
        dropout_rate (float, optional): The probability of an element to be zeroed 
            in the Dropout layer. Used for regularization to prevent overfitting. 
            Defaults to 0.3.

    Returns:
        DenseNet121: An initialized PyTorch model instance (subclass of nn.Module), 
            ready to be moved to a device (CPU/GPU) for training or inference.
    """
    
    model = DenseNet121(num_classes=num_classes, dropout_rate=dropout_rate)
    return model

In [7]:
class MobileNetV2(nn.Module):
    """
    MobileNetV2-based model for image classification using Transfer Learning.
    
    Architecture:
    - Backbone: MobileNetV2 (frozen, ImageNet weights)
    - Head: Global Average Pooling -> Dropout -> Linear (Dense)
    
    Attributes:
        num_classes (int): Number of target classes.
        dropout_rate (float): Dropout probability.
    """
    
    def __init__(self, num_classes: int = NUM_CLASSES, dropout_rate: float = DROPOUT_RATE) -> None:
        super().__init__()

        # Load Pre-trained MobileNetV2
        weights = models.MobileNet_V2_Weights.IMAGENET1K_V1
        backbone = models.mobilenet_v2(weights=weights)

        # MobileNetV2 .features contains all the convolutional layers (Inverted Residuals)
        self.features = backbone.features

        # Unfreeze ALL
        for param in self.features.parameters():
            param.requires_grad = True

        # Define Custom Classifier Head
        # MobileNetV2 output feature map has 1280 channels
        self.in_features = 1280 
        
        self.global_avg_pool = nn.AdaptiveAvgPool2d(output_size=1) # Reduces (Batch, 1280, H, W) -> (Batch, 1280, 1, 1)
        self.flatten = nn.Flatten(start_dim=1) # Flatten (Batch, 1280, 1, 1) -> (Batch, 1280)
        self.dropout = nn.Dropout(p=dropout_rate)
        self.classifier = nn.Linear(in_features=self.in_features, out_features=num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass of the model.

        Args:
            x (torch.Tensor): Input image batch (B, C, H, W)

        Returns:
            torch.Tensor: Raw logits (B, num_classes). 
                          NOTE: Does NOT apply Softmax.
        """

        # Feature extraction (Frozen)
        x = self.features(x)

        # Classification Head (Trainable)
        x = self.global_avg_pool(x)
        x = self.flatten(x)
        x = self.dropout(x)

        # Output Logits
        logits = self.classifier(x)

        return logits

In [8]:
def build_mobilenetv2(num_classes: int = NUM_CLASSES, dropout_rate: float = DROPOUT_RATE) -> MobileNetV2:
    """
    Factory function to instantiate the customized MobileNetV2 model for Transfer Learning.

    This function initializes a `MobileNetV2` which includes:
    1. A frozen MobileNetV2 backbone (pre-trained on ImageNet).
    2. A custom trainable classification head (Pooling -> Dropout -> Linear).

    Args:
        num_classes (int, optional): The number of target classes for the classification task. 
            Determines the output size of the final Linear layer. Defaults to 3.
        dropout_rate (float, optional): The probability of an element to be zeroed 
            in the Dropout layer. Used for regularization to prevent overfitting. 
            Defaults to 0.3.

    Returns:
        MobileNetV2: An initialized PyTorch model instance (subclass of nn.Module), 
            ready to be moved to a device (CPU/GPU) for training or inference.
    """

    model = MobileNetV2(num_classes=num_classes, dropout_rate=dropout_rate)
    return model

In [9]:
class ResNet50(nn.Module):
    """
    ResNet50-based model for image classification using Transfer Learning.
    
    Architecture:
    - Backbone: ResNet50 (frozen, ImageNet weights)
    - Head: Global Average Pooling -> Dropout -> Linear (Dense)
    
    Attributes:
        num_classes (int): Number of target classes.
        dropout_rate (float): Dropout probability.
    """
    
    def __init__(self, num_classes: int = 3, dropout_rate: float = 0.3) -> None:
        super().__init__()

        # Load Pre-trained ResNet50
        weights = models.ResNet50_Weights.IMAGENET1K_V1
        original_model = models.resnet50(weights=weights)

        # Feature Extractor
        # ResNet50 structure: [conv1, bn1, ..., layer1, layer2, layer3, layer4, avgpool, fc]
        # We remove the last 2 layers ('avgpool' and 'fc') to keep only the convolutional part.
        self.features = nn.Sequential(*list(original_model.children())[:-2])

        # Unfreeze ALL
        for param in self.features.parameters():
            param.requires_grad = True

        # 4. Define Custom Classifier Head
        # ResNet50's final conv block (layer4) outputs 2048 channels.
        self.in_features = 2048 
        
        self.global_avg_pool = nn.AdaptiveAvgPool2d(output_size=1) # Reduces (Batch, 2048, H, W) -> (Batch, 2048, 1, 1)
        self.flatten = nn.Flatten(start_dim=1) # Flatten (Batch, 2048, 1, 1) -> (Batch, 2048)
        self.dropout = nn.Dropout(p=dropout_rate)
        self.classifier = nn.Linear(in_features=self.in_features, out_features=num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass of the model.

        Args:
            x (torch.Tensor): Input image batch (B, C, H, W)

        Returns:
            torch.Tensor: Raw logits (B, num_classes). 
                          NOTE: Does NOT apply Softmax.
        """

        # Feature extraction (Frozen)
        x = self.features(x)

        # Classification Head (Trainable)
        x = self.global_avg_pool(x)
        x = self.flatten(x)
        x = self.dropout(x)

        # Output Logits
        logits = self.classifier(x)

        return logits

In [10]:
def build_resnet50(num_classes: int = NUM_CLASSES, dropout_rate: float = DROPOUT_RATE) -> ResNet50:
    """
    Factory function to instantiate the customized ResNet50 model for Transfer Learning.

    This function initializes a `ResNet50` which includes:
    1. A frozen ResNet50 backbone (pre-trained on ImageNet).
    2. A custom trainable classification head (Pooling -> Dropout -> Linear).

    Args:
        num_classes (int, optional): The number of target classes for the classification task. 
            Determines the output size of the final Linear layer. Defaults to 3.
        dropout_rate (float, optional): The probability of an element to be zeroed 
            in the Dropout layer. Used for regularization to prevent overfitting. 
            Defaults to 0.3.

    Returns:
        ResNet50: An initialized PyTorch model instance (subclass of nn.Module), 
            ready to be moved to a device (CPU/GPU) for training or inference.
    """

    model = ResNet50(num_classes=num_classes, dropout_rate=dropout_rate)
    return model

In [11]:
class VGG19(nn.Module):
    """
    VGG19-based model for image classification using Transfer Learning.
    
    Architecture:
    - Backbone: VGG19 (frozen, ImageNet weights)
    - Head: Global Average Pooling -> Dropout -> Linear (Dense)
    
    Attributes:
        num_classes (int): Number of target classes.
        dropout_rate (float): Dropout probability.
    """
    
    def __init__(self, num_classes: int = NUM_CLASSES, dropout_rate: float = DROPOUT_RATE) -> None:
        super().__init__()

        # Load Pre-trained VGG19
        weights = models.VGG19_Weights.IMAGENET1K_V1
        backbone = models.vgg19(weights=weights)

        # VGG19 .features contains all Conv/Relu/MaxPool layers
        self.features = backbone.features

        # Unfreeze ALL
        for param in self.features.parameters():
            param.requires_grad = True

        # Define Custom Classifier Head
        self.global_avg_pool = nn.AdaptiveAvgPool2d(output_size=1) # Reduces (Batch, 512, H, W) -> (Batch, 512, 1, 1)
        self.flatten = nn.Flatten(start_dim=1) # Flatten (Batch, 512, 1, 1) -> (Batch, 512)
        self.dropout = nn.Dropout(p=dropout_rate)
        self.classifier = nn.Linear(in_features=512, out_features=num_classes) # VGG19 features output exactly 512 channels

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass of the model.

        Args:
            x (torch.Tensor): Input image batch (B, C, H, W)

        Returns:
            torch.Tensor: Raw logits (B, num_classes). 
                          NOTE: Does NOT apply Softmax.
        """

        # Feature extraction (Frozen)
        x = self.features(x)

        # Classification Head (Trainable)
        x = self.global_avg_pool(x)
        x = self.flatten(x)
        x = self.dropout(x)

        # Output Logits
        logits = self.classifier(x)

        return logits

In [12]:
def build_vgg19(num_classes: int = NUM_CLASSES, dropout_rate: float = DROPOUT_RATE) -> VGG19:
    """
    Factory function to instantiate the customized VGG19 model for Transfer Learning.

    This function initializes a `VGG19` which includes:
    1. A frozen VGG19 backbone (pre-trained on ImageNet).
    2. A custom trainable classification head (Pooling -> Dropout -> Linear).

    Args:
        num_classes (int, optional): The number of target classes for the classification task. 
            Determines the output size of the final Linear layer. Defaults to 3.
        dropout_rate (float, optional): The probability of an element to be zeroed 
            in the Dropout layer. Used for regularization to prevent overfitting. 
            Defaults to 0.3.

    Returns:
        VGG19Classifier: An initialized PyTorch model instance (subclass of nn.Module), 
            ready to be moved to a device (CPU/GPU) for training or inference.
    """

    model = VGG19(num_classes=num_classes, dropout_rate=dropout_rate)
    return model

In [13]:
def build_edgenext_xxs_model(num_classes: int = 3) -> nn.Module:
    """Builds a lightweight Student Model (EdgeNeXt-XXS) for distillation.
    
    This function instantiates the Extra-Extra-Small variant of EdgeNeXt via `timm`. 
    Introduced in ECCV 2022, EdgeNeXt is a state-of-the-art hybrid architecture 
    that amalgamates CNNs and Vision Transformers. It employs Split Depth-wise 
    Transposed Attention (SDTA) to effectively capture global context while 
    minimizing the computational overhead typically associated with ViTs.

    Args:
        num_classes (int, optional): Number of output classes for the 
            classification head. Defaults to 3.

    Returns:
        nn.Module: The initialized EdgeNeXt-XXS PyTorch model (~1.3M params).
    """
    print("Initializing EdgeNeXt-XXS student model...")
    
    # timm seamlessly integrates pre-trained weights and adapts the classifier
    model = timm.create_model(
        model_name='edgenext_xx_small', 
        pretrained=True, 
        num_classes=num_classes
    )
    
    return model

In [14]:
def build_mobilevit_xxs_model(num_classes: int = 3) -> nn.Module:
    """Builds a lightweight Student Model (MobileViT-XXS) for distillation.
    
    This function instantiates the Extra-Extra-Small (XXS) variant of MobileViT 
    using the `timm` library. MobileViT is a hybrid architecture that seamlessly 
    combines the spatial inductive biases of Convolutional Neural Networks (CNNs) 
    with the global attention mechanisms of Vision Transformers (ViTs).

    Args:
        num_classes (int, optional): Number of output classes for the 
            classification head. Defaults to 3.

    Returns:
        nn.Module: The initialized MobileViT-XXS PyTorch model (~1.2M params).
    """
    print("Initializing MobileViT-XXS student model...")
    
    # The timm library automatically downloads the pre-trained ImageNet weights 
    # and safely replaces the final classification head to match `num_classes`.
    model = timm.create_model(
        model_name='mobilevit_xxs', 
        pretrained=True, 
        num_classes=num_classes
    )
    
    return model

In [15]:
def get_tta_transform() -> v2.Compose:
    """Constructs the Test-Time Augmentation (TTA) pipeline.
    
    Returns:
        v2.Compose: A composition of torchvision transforms.
    """
    return v2.Compose([
        v2.RandomHorizontalFlip(p=0.5),
        v2.ColorJitter(brightness=0.1, contrast=(0.9, 1.1))
    ])


class HeavyTeacherPipeline:
    """Simulates the inference pipeline of the Heavy Teacher ensemble with TTA.
    
    Attributes:
        models (List[nn.Module]): List of loaded PyTorch models.
        tta_rounds (int): Number of Test-Time Augmentation iterations.
        tta_transform (v2.Compose): The augmentation transformations.
        device (str): Computation device ('cuda' or 'cpu').
    """
    
    def __init__(self, models_list: List[nn.Module], tta_rounds: int = 5, num_classes: int = NUM_CLASSES, device: str = 'cuda') -> None:
        """Initializes the pipeline with the given models and TTA configuration."""
        self.models = [model.to(device).eval() for model in models_list]
        self.tta_rounds = tta_rounds
        self.tta_transform = get_tta_transform()
        self.num_classes = num_classes  # Lưu linh hoạt số lượng class
        self.device = device

    def predict(self, x: torch.Tensor) -> torch.Tensor:
        """Performs ensemble inference with TTA on a single input tensor.
        
        Args:
            x (torch.Tensor): The input image tensor of shape (1, C, H, W).
            
        Returns:
            torch.Tensor: The final averaged probability distribution across 
                all models and TTA rounds.
        """
        final_probs = torch.zeros((1, self.num_classes), device=self.device)
        
        with torch.no_grad():
            for model in self.models:
                model_probs = torch.zeros((1, self.num_classes), device=self.device)
                
                # Execute TTA iterations
                for _ in range(self.tta_rounds):
                    augmented_x = self.tta_transform(x)
                    logits = model(augmented_x)
                    probs = torch.softmax(logits, dim=1)
                    model_probs += probs
                    
                model_probs /= self.tta_rounds
                final_probs += model_probs
                
        final_probs /= len(self.models)
        return final_probs

In [16]:
def measure_inference_speed(
    model_pipeline: Union[nn.Module, HeavyTeacherPipeline], 
    model_name: str, 
    device: str = 'cuda', 
    input_size: Tuple[int, int, int, int] = (1, 3, 224, 224), 
    repetitions: int = 300
) -> Tuple[float, float]:
    """Measures the latency and throughput (FPS) of a given model or pipeline.
    
    Includes a warm-up phase to initialize GPU clocks and relies on 
    torch.cuda.synchronize() for accurate timekeeping.
    
    Args:
        model_pipeline (Union[nn.Module, HeavyTeacherPipeline]): The model or ensemble.
        model_name (str): Identifier for logging purposes.
        device (str): Computation device ('cuda' or 'cpu').
        input_size (Tuple[int, int, int, int]): Dimensions of the dummy input tensor.
        repetitions (int): Number of forward passes to average over.
        
    Returns:
        Tuple[float, float]: Mean latency (in ms) and throughput (in FPS).
    """
    dummy_input = torch.randn(input_size, device=device)
    
    print(f"\n[{model_name}] Initiating GPU warm-up sequence...")
    with torch.no_grad():
        for _ in range(100):
            if isinstance(model_pipeline, HeavyTeacherPipeline):
                _ = model_pipeline.predict(dummy_input)
            else:
                _ = model_pipeline(dummy_input)
                
    print(f"[{model_name}] Executing official measurement over {repetitions} iterations...")
    timings = np.zeros((repetitions, 1))
    
    with torch.no_grad():
        for rep in range(repetitions):
            if device == 'cuda':
                torch.cuda.synchronize()
            
            start_time = time.perf_counter()
            
            # Forward pass execution
            if isinstance(model_pipeline, HeavyTeacherPipeline):
                _ = model_pipeline.predict(dummy_input)
            else:
                _ = model_pipeline(dummy_input)
                
            if device == 'cuda':
                torch.cuda.synchronize()
                
            end_time = time.perf_counter()
            timings[rep] = (end_time - start_time) * 1000  # Convert to milliseconds
            
    mean_lat = float(np.mean(timings))
    std_lat = float(np.std(timings))
    fps = 1000.0 / mean_lat
    
    print(f"INFERENCE PROFILING REPORT: {model_name}")
    print(f"Latency:    {mean_lat:.2f} ms ± {std_lat:.2f} ms")
    print(f"Throughput: {fps:.2f} FPS")
    
    return mean_lat, fps

In [17]:
def main_profiling() -> None:
    """Main execution block to instantiate models and run speed profiling."""
    
    # Instantiate core architectures (No pre-trained weights needed for speed profiling)
    densenet121_model = build_densenet121().to(DEVICE).eval()
    mobilenetv2_model = build_mobilenetv2().to(DEVICE).eval()
    resnet50_model = build_resnet50().to(DEVICE).eval()

    # Construct the Heavy Teacher Pipeline
    teacher_pipeline = HeavyTeacherPipeline(
        models_list=[densenet121_model, mobilenetv2_model, resnet50_model],
        tta_rounds=5,
        device=DEVICE
    )

    # Teacher
    measure_inference_speed(
        model_pipeline=teacher_pipeline, 
        model_name="Heavy Teacher",
        device=DEVICE
    )

    # Baseline Single
    measure_inference_speed(densenet121_model.to(DEVICE).eval() ,"Single Baseline: DenseNet121", device=DEVICE)
    measure_inference_speed(mobilenetv2_model.to(DEVICE).eval(),"Single Baseline: MobileNetV2", device=DEVICE)
    measure_inference_speed(resnet50_model.to(DEVICE).eval(), "Single Baseline: ResNet50", device=DEVICE)


    # EdgeneXt xxs
    edgenext_xxs_model = build_edgenext_xxs_model().to(DEVICE).eval()
    measure_inference_speed(edgenext_xxs_model, "EdgeNeXt-XXS (Student)", device=DEVICE)

    # MobileViT xxs
    mobilevit_xxs_model = build_mobilevit_xxs_model().to(DEVICE).eval()
    measure_inference_speed(mobilevit_xxs_model, "MobileVit-XXS (Student)", device=DEVICE)
    


In [18]:
main_profiling()

Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 167MB/s] 


Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 121MB/s]


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


...


[Heavy Teacher] Initiating GPU warm-up sequence...
[Heavy Teacher] Executing official measurement over 300 iterations...
INFERENCE PROFILING REPORT: Heavy Teacher
Latency:    120.26 ms ± 4.45 ms
Throughput: 8.32 FPS

[Single Baseline: DenseNet121] Initiating GPU warm-up sequence...
[Single Baseline: DenseNet121] Executing official measurement over 300 iterations...
INFERENCE PROFILING REPORT: Single Baseline: DenseNet121
Latency:    13.06 ms ± 0.98 ms
Throughput: 76.57 FPS

[Single Baseline: MobileNetV2] Initiating GPU warm-up sequence...
[Single Baseline: MobileNetV2] Executing official measurement over 300 iterations...
INFERENCE PROFILING REPORT: Single Baseline: MobileNetV2
Latency:    4.42 ms ± 0.20 ms
Throughput: 226.25 FPS

[Single Baseline: ResNet50] Initiating GPU warm-up sequence...
[Single Baseline: ResNet50] Executing official measurement over 300 iterations...
INFERENCE PROFILING REPORT: Single Baseline: ResNet50
Latency:    5.78 ms ± 0.20 ms
Throughput: 172.92 FPS
Initia

model.safetensors:   0%|          | 0.00/5.32M [00:00<?, ?B/s]


[EdgeNeXt-XXS (Student)] Initiating GPU warm-up sequence...
[EdgeNeXt-XXS (Student)] Executing official measurement over 300 iterations...
INFERENCE PROFILING REPORT: EdgeNeXt-XXS (Student)
Latency:    6.59 ms ± 0.54 ms
Throughput: 151.78 FPS
Initializing MobileViT-XXS student model...


model.safetensors:   0%|          | 0.00/5.14M [00:00<?, ?B/s]


[MobileVit-XXS (Student)] Initiating GPU warm-up sequence...
[MobileVit-XXS (Student)] Executing official measurement over 300 iterations...
INFERENCE PROFILING REPORT: MobileVit-XXS (Student)
Latency:    7.91 ms ± 0.31 ms
Throughput: 126.47 FPS
